# Notebook 4: Głębokie sieci neuronowe (Deep Learning)

## Cel
Implementacja i porównanie 3 architektur głębokich sieci neuronowych:
1. **CNN1D** – prosta sieć konwolucyjna 1D (3 bloki konwolucyjne + 2 FC)
2. **ResNet1D** – głęboka sieć rezydualna (He et al., 2016, adaptacja 1D)
3. **BiLSTM** – dwukierunkowa sieć LSTM z mechanizmem uwagi (attention)

Dane wejściowe: surowe przetworzone sygnały EKG (1000×12) → (12, 1000) dla Conv1d.
Ocena bez strojenia hiperparametrów (Kontrola 2).

**Wymaganie:** Kontrola pośrednia nr 2 – wszystkie architektury DL (bez optymalizacji)


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score, ConfusionMatrixDisplay,
)

sys.path.insert(0, '..')
from src.data_loader import (
    load_ptbxl_metadata, build_labels, load_raw_data,
    get_train_val_test_split, SUPERCLASSES,
)
from src.preprocessing import preprocess_batch
from src.deep_models import (
    ECGDataset, CNN1D, ResNet1D, BiLSTMClassifier,
    Trainer, get_device, get_deep_models,
)

plt.rcParams['figure.dpi'] = 100
sns.set_theme(style='whitegrid')

DATA_PATH = '../data/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
FS = 100
DEVICE = get_device()

print(f"PyTorch {torch.__version__}")
print(f"Urządzenie obliczeniowe: {DEVICE}")


## 1. Przygotowanie danych

Wczytujemy przetworzone sygnały EKG i tworzymy PyTorch DataLoader'y.


In [ ]:
# Wczytaj metadane i etykiety
Y = load_ptbxl_metadata(DATA_PATH)
Y = build_labels(Y, DATA_PATH)
train_idx, val_idx, test_idx = get_train_val_test_split(Y)

# ── Podzbiór do demonstracji ────────────────────────────────────────────────
# Zmień N_PER_CLASS=None dla pełnego zbioru (trening ~1-2h na CPU)
N_PER_CLASS = 80   # 80 przykładów na klasę w treningu

def sample_per_class(df, idx, n=None):
    sampled = []
    for cls in SUPERCLASSES:
        cls_idx = df[(df.label_single == cls) & df.index.isin(idx)].index
        sampled.extend(cls_idx[:n] if n else cls_idx)
    return df.loc[sampled]

Y_train_s = sample_per_class(Y, train_idx, N_PER_CLASS)
Y_val_s   = sample_per_class(Y, val_idx,   N_PER_CLASS // 4)
Y_test_s  = sample_per_class(Y, test_idx,  N_PER_CLASS // 4)

print(f"Trening:   {len(Y_train_s):,} | Walidacja: {len(Y_val_s):,} | Test: {len(Y_test_s):,}")


In [ ]:
# Wczytaj surowe sygnały
print("Ładowanie i przetwarzanie sygnałów...")
def load_and_process(Y_df):
    X_raw = load_raw_data(Y_df, sampling_rate=FS, data_path=DATA_PATH)
    X_proc = preprocess_batch(X_raw, fs=FS, verbose=False)
    return X_proc

X_train = load_and_process(Y_train_s)
X_val   = load_and_process(Y_val_s)
X_test  = load_and_process(Y_test_s)

# Koduj etykiety
le = LabelEncoder()
le.fit(SUPERCLASSES)
y_train = le.transform(Y_train_s['label_single'].values)
y_val   = le.transform(Y_val_s['label_single'].values)
y_test  = le.transform(Y_test_s['label_single'].values)

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"Klasy: {le.classes_}")


In [ ]:
# Utwórz DataLoader'y PyTorch
BATCH_SIZE = 32

train_dataset = ECGDataset(X_train, y_train)
val_dataset   = ECGDataset(X_val,   y_val)
test_dataset  = ECGDataset(X_test,  y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sprawdź kształt batcha
X_batch, y_batch = next(iter(train_loader))
print(f"Kształt batcha wejściowego: {X_batch.shape}")
print(f"  (batch={X_batch.shape[0]}, leads={X_batch.shape[1]}, samples={X_batch.shape[2]})")
print(f"Kształt etykiet: {y_batch.shape}")


## 2. Architektura 1: CNN1D

Prosta sieć konwolucyjna 1D złożona z 3 bloków konwolucyjnych,
z BatchNorm, ReLU i MaxPool po każdym bloku.


In [ ]:
# Inicjalizacja modelu CNN1D
cnn_model = CNN1D(num_channels=12, num_classes=len(SUPERCLASSES))

# Wyświetl architekturę
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN1D – liczba parametrów: {total_params:,}")
print()
print(cnn_model)


In [ ]:
# Trening CNN1D
print("\n=== Trening CNN1D ===")
EPOCHS = 20
LR = 1e-3

cnn_trainer = Trainer(cnn_model, device=DEVICE, learning_rate=LR)
cnn_history = cnn_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 3. Architektura 2: ResNet1D

Głęboka sieć rezydualna adaptowana dla sygnałów 1D. Połączenia skip-connection
(Skip connections / Residual connections) umożliwiają trening bardzo głębokich
sieci bez problemu zanikającego gradientu.


In [ ]:
# Inicjalizacja modelu ResNet1D
resnet_model = ResNet1D(num_channels=12, num_classes=len(SUPERCLASSES))

total_params = sum(p.numel() for p in resnet_model.parameters() if p.requires_grad)
print(f"ResNet1D – liczba parametrów: {total_params:,}")
print()
print(resnet_model)


In [ ]:
# Trening ResNet1D
print("\n=== Trening ResNet1D ===")
resnet_trainer = Trainer(resnet_model, device=DEVICE, learning_rate=LR)
resnet_history = resnet_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 4. Architektura 3: Bidirectional LSTM

Dwukierunkowa sieć LSTM z mechanizmem uwagi (attention). LSTM modeluje
długoterminowe zależności w sygnale; mechanizm uwagi pozwala sieci skupić się
na najważniejszych momentach czasowych (np. kompleksach QRS).


In [ ]:
# Inicjalizacja modelu BiLSTM
lstm_model = BiLSTMClassifier(
    input_size=12, hidden_size=128,
    num_layers=2, num_classes=len(SUPERCLASSES)
)

total_params = sum(p.numel() for p in lstm_model.parameters() if p.requires_grad)
print(f"BiLSTM – liczba parametrów: {total_params:,}")
print()
print(lstm_model)


In [ ]:
# Trening BiLSTM
print("\n=== Trening BiLSTM ===")
lstm_trainer = Trainer(lstm_model, device=DEVICE, learning_rate=LR)
lstm_history = lstm_trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=True)


## 5. Krzywe uczenia (Loss i Accuracy)

In [ ]:
# Wizualizacja krzywych uczenia dla wszystkich 3 modeli
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_hist = {
    'CNN1D':    (cnn_history,    'steelblue'),
    'ResNet1D': (resnet_history, 'darkorange'),
    'BiLSTM':   (lstm_history,   'green'),
}

# Strata (Loss)
for name, (hist, color) in models_hist.items():
    axes[0].plot(hist['train_loss'], '--', color=color, alpha=0.7, label=f'{name} train')
    axes[0].plot(hist['val_loss'],   '-',  color=color, linewidth=2, label=f'{name} val')

axes[0].set_title('Krzywe straty (Loss)', fontweight='bold')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].legend(fontsize=8)

# Dokładność (Accuracy)
for name, (hist, color) in models_hist.items():
    axes[1].plot(hist['val_acc'], '-', color=color, linewidth=2, label=name)

axes[1].set_title('Dokładność walidacyjna (Val Accuracy)', fontweight='bold')
axes[1].set_xlabel('Epoka')
axes[1].set_ylabel('Accuracy')
axes[1].legend()

plt.tight_layout()
plt.savefig('../results/deep_learning_training_curves.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Ewaluacja na zbiorze testowym

In [ ]:
# Ewaluacja wszystkich modeli na zbiorze testowym
results_dl = {}

for name, trainer in [
    ('CNN1D', cnn_trainer),
    ('ResNet1D', resnet_trainer),
    ('BiLSTM', lstm_trainer)
]:
    y_true, y_pred = trainer.predict(test_loader)
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1w = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    cm  = confusion_matrix(y_true, y_pred)

    results_dl[name] = {'y_true': y_true, 'y_pred': y_pred,
                        'accuracy': acc, 'f1_macro': f1m, 'f1_weighted': f1w,
                        'confusion_matrix': cm}

    print(f"\n{'='*50}")
    print(f"Model: {name}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"F1 Macro:    {f1m:.4f}")
    print(f"F1 Weighted: {f1w:.4f}")
    print(classification_report(y_true, y_pred, target_names=le.classes_, zero_division=0))


In [ ]:
# Macierze pomyłek DL
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for ax, (name, result) in zip(axes, results_dl.items()):
    disp = ConfusionMatrixDisplay(result['confusion_matrix'], display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={result["accuracy"]:.3f} | F1={result["f1_macro"]:.3f}',
                 fontweight='bold')

plt.tight_layout()
plt.savefig('../results/deep_learning_confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# Zbiorcze porównanie modeli DL
print("\n=== PORÓWNANIE ARCHITEKTUR DEEP LEARNING ===")
summary_dl = pd.DataFrame([
    {
        'Model': name,
        'Accuracy': r['accuracy'],
        'F1 Macro': r['f1_macro'],
        'F1 Weighted': r['f1_weighted'],
        'Parametrów': sum(p.numel() for p in m.parameters() if p.requires_grad),
    }
    for (name, r), m in zip(
        results_dl.items(),
        [cnn_model, resnet_model, lstm_model]
    )
])
print(summary_dl.to_string(index=False))


In [ ]:
# Zapis modeli
os.makedirs('../models', exist_ok=True)

torch.save(cnn_model.state_dict(),    '../models/cnn1d.pt')
torch.save(resnet_model.state_dict(), '../models/resnet1d.pt')
torch.save(lstm_model.state_dict(),   '../models/bilstm.pt')

print("Zapisano modele:")
print("  models/cnn1d.pt")
print("  models/resnet1d.pt")
print("  models/bilstm.pt")


In [ ]:
print("\n=== PODSUMOWANIE – DEEP LEARNING ===")
print(f"\nZaimplementowane architektury:")
print(f"  1. CNN1D    – prosta sieć konwolucyjna 1D")
print(f"  2. ResNet1D – sieć rezydualna (skip connections)")
print(f"  3. BiLSTM   – dwukierunkowy LSTM z mechanizmem uwagi")
print(f"\nTrening: {EPOCHS} epok, LR={LR}, batch={BATCH_SIZE}")
print(f"Optymalizator: Adam + ReduceLROnPlateau")
print(f"Regularyzacja: Dropout, BatchNorm, Weight Decay, Gradient Clipping")


---
## 7. Dostrajanie hiperparametrów sieci neuronowych

Dla każdej architektury przeszukujemy siatkę kluczowych hiperparametrów:
- **CNN1D, ResNet1D:** `learning_rate` × `batch_size`
- **BiLSTM:** `learning_rate` × `hidden_size`

Każda konfiguracja trenowana jest przez **10 epok** (szybki screening).
Następnie najlepsza konfiguracja każdego modelu trenowana jest **pełne 20 epok**.

Wizualizacje:
- **Heatmapy** – val_accuracy dla każdej kombinacji parametrów
- **Krzywe uczenia** – porównanie najlepszych LR dla każdego modelu
- **Porównanie przed/po** – domyślna konfiguracja vs optymalna

In [ ]:
from src.deep_models import tune_dl_hyperparameters

# ── Uruchom przeszukiwanie siatki ───────────────────────────────────────────
# TUNE_EPOCHS=10 – krótki trening dla szybkiego porównania konfiguracji.
# Dla bardziej wiarygodnych wyników zwiększ do 20-30.
TUNE_EPOCHS = 10

print(f"Przeszukiwanie siatki hiperparametrów ({TUNE_EPOCHS} epok per konfiguracja)...")
print(f"Urządzenie: {DEVICE}\n")

dl_tune_results = tune_dl_hyperparameters(
    X_train, y_train,
    X_val,   y_val,
    tune_epochs=TUNE_EPOCHS,
    device=DEVICE,
)

print("\n✓ Przeszukiwanie zakończone.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 1 i 2: Heatmapy CNN1D i ResNet1D (lr × batch_size)
# ═══════════════════════════════════════════════════════════════════════════
# Heatmapa pokazuje jak kombinacja learning_rate i batch_size wpływa
# na najlepszą dokładność walidacyjną osiągniętą w ciągu TUNE_EPOCHS epok.
# Czerwone pole = najlepsza konfiguracja.

lr_values   = [1e-4, 5e-4, 1e-3, 5e-3]
batch_sizes  = [16, 32, 64]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, arch_name, cmap in [
    (axes[0], 'CNN1D',    'YlOrRd'),
    (axes[1], 'ResNet1D', 'Blues'),
]:
    results = dl_tune_results[arch_name]['all_results']
    # Buduj macierz: wiersze = lr, kolumny = batch_size
    matrix = np.zeros((len(lr_values), len(batch_sizes)))
    for r in results:
        i = lr_values.index(r['lr'])
        j = batch_sizes.index(r['batch_size'])
        matrix[i, j] = r['val_acc']

    im = ax.imshow(matrix, cmap=cmap, aspect='auto',
                   vmin=matrix.min() - 0.02, vmax=matrix.max() + 0.01)
    plt.colorbar(im, ax=ax, label='Val Accuracy')

    ax.set_xticks(range(len(batch_sizes)))
    ax.set_yticks(range(len(lr_values)))
    ax.set_xticklabels(batch_sizes, fontsize=11)
    ax.set_yticklabels([f'{lr:.0e}' for lr in lr_values], fontsize=11)
    ax.set_xlabel('batch_size', fontsize=12)
    ax.set_ylabel('learning_rate', fontsize=12)

    # Annotate każdą komórkę
    for i in range(len(lr_values)):
        for j in range(len(batch_sizes)):
            ax.text(j, i, f'{matrix[i,j]:.3f}',
                    ha='center', va='center', fontsize=10, fontweight='bold',
                    color='white' if matrix[i,j] > matrix.max() - 0.03 else 'black')

    # Zaznacz najlepszą komórkę czerwonym konturem
    best = dl_tune_results[arch_name]['best_config']
    bi   = lr_values.index(best['lr'])
    bj   = batch_sizes.index(best['batch_size'])
    ax.add_patch(plt.Rectangle((bj-0.5, bi-0.5), 1, 1,
                                fill=False, edgecolor='red', linewidth=3))

    ax.set_title(
        f"{arch_name}: lr × batch_size → Val Accuracy\n"
        f"Najlepsze: lr={best['lr']:.0e}, bs={best['batch_size']} "
        f"→ acc={best['val_acc']:.4f}",
        fontweight='bold'
    )

plt.tight_layout()
plt.savefig('../results/tuning_dl_cnn_resnet_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 3: Heatmapa BiLSTM (lr × hidden_size)
# ═══════════════════════════════════════════════════════════════════════════
# BiLSTM ma dodatkowy kluczowy parametr: hidden_size (rozmiar warstwy ukrytej).
# Większy hidden_size = większa pojemność modelu, ale wolniejszy trening.

hidden_sizes = [64, 128, 256]

lstm_results = dl_tune_results['BiLSTM']['all_results']
matrix_lstm  = np.zeros((len(lr_values), len(hidden_sizes)))

for r in lstm_results:
    i = lr_values.index(r['lr'])
    j = hidden_sizes.index(r['hidden_size'])
    matrix_lstm[i, j] = r['val_acc']

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(matrix_lstm, cmap='Greens', aspect='auto',
               vmin=matrix_lstm.min() - 0.02, vmax=matrix_lstm.max() + 0.01)
plt.colorbar(im, ax=ax, label='Val Accuracy')

ax.set_xticks(range(len(hidden_sizes)))
ax.set_yticks(range(len(lr_values)))
ax.set_xticklabels(hidden_sizes, fontsize=11)
ax.set_yticklabels([f'{lr:.0e}' for lr in lr_values], fontsize=11)
ax.set_xlabel('hidden_size', fontsize=12)
ax.set_ylabel('learning_rate', fontsize=12)

for i in range(len(lr_values)):
    for j in range(len(hidden_sizes)):
        ax.text(j, i, f'{matrix_lstm[i,j]:.3f}',
                ha='center', va='center', fontsize=11, fontweight='bold',
                color='white' if matrix_lstm[i,j] > matrix_lstm.max() - 0.03 else 'black')

# Zaznacz najlepszą konfigurację
best_lstm_cfg = dl_tune_results['BiLSTM']['best_config']
bi = lr_values.index(best_lstm_cfg['lr'])
bj = hidden_sizes.index(best_lstm_cfg['hidden_size'])
ax.add_patch(plt.Rectangle((bj-0.5, bi-0.5), 1, 1,
                             fill=False, edgecolor='red', linewidth=3))

ax.set_title(
    f"BiLSTM: lr × hidden_size → Val Accuracy\n"
    f"Najlepsze: lr={best_lstm_cfg['lr']:.0e}, "
    f"hidden={best_lstm_cfg['hidden_size']} "
    f"→ acc={best_lstm_cfg['val_acc']:.4f}",
    fontweight='bold'
)

plt.tight_layout()
plt.savefig('../results/tuning_dl_bilstm_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 4: Krzywe uczenia – porównanie różnych LR dla CNN1D
# ═══════════════════════════════════════════════════════════════════════════
# Pokazujemy jak różne learning_rate wpływają na trajektorię uczenia.
# Używamy wyników z przeszukiwania (batch_size=32 jako reprezentatywny).

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors_lr = ['navy', 'steelblue', 'darkorange', 'red']

for ax, arch_name in zip(axes, ['CNN1D', 'ResNet1D', 'BiLSTM']):
    histories = dl_tune_results[arch_name]['histories']
    best_cfg  = dl_tune_results[arch_name]['best_config']

    # Dla CNN1D/ResNet1D: porównaj LR przy bs=32
    # Dla BiLSTM: porównaj LR przy hidden_size=128
    fixed_val = 32 if arch_name in ('CNN1D', 'ResNet1D') else 128
    fixed_key = 'bs' if arch_name in ('CNN1D', 'ResNet1D') else 'hs'

    for lr, color in zip(lr_values, colors_lr):
        label = f"lr={lr:.0e}_{fixed_key}={fixed_val}"
        if label in histories:
            accs = histories[label]
            is_best = (lr == best_cfg['lr'])
            ax.plot(range(1, len(accs)+1), accs,
                    color=color,
                    linewidth=3 if is_best else 1.5,
                    linestyle='-' if is_best else '--',
                    label=f'lr={lr:.0e}' + (' ★' if is_best else ''),
                    alpha=1.0 if is_best else 0.6)

    ax.set_title(f'{arch_name}: wpływ learning_rate\n({"bs" if arch_name != "BiLSTM" else "hidden"}={fixed_val})', fontweight='bold')
    ax.set_xlabel('Epoka')
    ax.set_ylabel('Val Accuracy')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle(f'Krzywe uczenia dla różnych learning_rate ({TUNE_EPOCHS} epok)',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/tuning_dl_lr_curves.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Pełny trening z najlepszymi hiperparametrami (20 epok)
# ═══════════════════════════════════════════════════════════════════════════
# Po znalezieniu najlepszych konfiguracji trenujemy pełne modele,
# które zostaną porównane z domyślnymi (z sekcji 2-4).

print("Pełny trening z optymalnymi hiperparametrami (20 epok)...")

tuned_models   = {}
tuned_trainers = {}
tuned_histories = {}

# ── CNN1D (optymalne) ────────────────────────────────────────────────────
best_cnn_cfg = dl_tune_results['CNN1D']['best_config']
bs_cnn = best_cnn_cfg['batch_size']
lr_cnn = best_cnn_cfg['lr']

cnn_tuned  = CNN1D(num_channels=12, num_classes=len(SUPERCLASSES))
tl_cnn = DataLoader(ECGDataset(X_train, y_train), batch_size=bs_cnn, shuffle=True,  num_workers=0)
vl_cnn = DataLoader(ECGDataset(X_val,   y_val),   batch_size=bs_cnn, shuffle=False, num_workers=0)

print(f"\n[CNN1D] lr={lr_cnn:.0e}, batch={bs_cnn}")
cnn_tuned_trainer = Trainer(cnn_tuned, device=DEVICE, learning_rate=lr_cnn)
tuned_histories['CNN1D'] = cnn_tuned_trainer.fit(tl_cnn, vl_cnn, epochs=EPOCHS, verbose=True)
tuned_models['CNN1D']    = cnn_tuned
tuned_trainers['CNN1D']  = cnn_tuned_trainer

# ── ResNet1D (optymalne) ─────────────────────────────────────────────────
best_resnet_cfg = dl_tune_results['ResNet1D']['best_config']
bs_rn = best_resnet_cfg['batch_size']
lr_rn = best_resnet_cfg['lr']

resnet_tuned = ResNet1D(num_channels=12, num_classes=len(SUPERCLASSES))
tl_rn = DataLoader(ECGDataset(X_train, y_train), batch_size=bs_rn, shuffle=True,  num_workers=0)
vl_rn = DataLoader(ECGDataset(X_val,   y_val),   batch_size=bs_rn, shuffle=False, num_workers=0)

print(f"\n[ResNet1D] lr={lr_rn:.0e}, batch={bs_rn}")
resnet_tuned_trainer = Trainer(resnet_tuned, device=DEVICE, learning_rate=lr_rn)
tuned_histories['ResNet1D'] = resnet_tuned_trainer.fit(tl_rn, vl_rn, epochs=EPOCHS, verbose=True)
tuned_models['ResNet1D']    = resnet_tuned
tuned_trainers['ResNet1D']  = resnet_tuned_trainer

# ── BiLSTM (optymalne) ───────────────────────────────────────────────────
best_lstm_cfg = dl_tune_results['BiLSTM']['best_config']
hs_lstm = best_lstm_cfg['hidden_size']
lr_lstm = best_lstm_cfg['lr']

lstm_tuned = BiLSTMClassifier(input_size=12, hidden_size=hs_lstm, num_classes=len(SUPERCLASSES))
tl_lstm = DataLoader(ECGDataset(X_train, y_train), batch_size=32, shuffle=True,  num_workers=0)
vl_lstm = DataLoader(ECGDataset(X_val,   y_val),   batch_size=32, shuffle=False, num_workers=0)

print(f"\n[BiLSTM] lr={lr_lstm:.0e}, hidden_size={hs_lstm}")
lstm_tuned_trainer = Trainer(lstm_tuned, device=DEVICE, learning_rate=lr_lstm)
tuned_histories['BiLSTM'] = lstm_tuned_trainer.fit(tl_lstm, vl_lstm, epochs=EPOCHS, verbose=True)
tuned_models['BiLSTM']    = lstm_tuned
tuned_trainers['BiLSTM']  = lstm_tuned_trainer

print("\n✓ Pełny trening zakończony.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 5: Porównanie PRZED / PO tuningu na zbiorze testowym
# ═══════════════════════════════════════════════════════════════════════════
# Oceniamy na zbiorze testowym: domyślne parametry (sekcja 2-4) vs optymalne.

print("Ewaluacja wyoptymalizowanych modeli na zbiorze testowym...")

test_loader_default = DataLoader(
    ECGDataset(X_test, y_test), batch_size=32, shuffle=False, num_workers=0
)

# Wyniki PRZED tuningiem (z sekcji 6)
f1_before = {name: results_dl[name]['f1_macro'] for name in ('CNN1D', 'ResNet1D', 'BiLSTM')}

# Wyniki PO tuningu
f1_after = {}
for arch_name, trainer in tuned_trainers.items():
    tl_test = DataLoader(
        ECGDataset(X_test, y_test),
        batch_size=dl_tune_results[arch_name]['best_config'].get('batch_size', 32),
        shuffle=False, num_workers=0
    )
    y_true, y_pred = trainer.predict(tl_test)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    f1_after[arch_name] = f1
    delta = f1 - f1_before[arch_name]
    print(f"  {arch_name:10s}: domyślne={f1_before[arch_name]:.4f}  optymalne={f1:.4f}  Δ={delta:+.4f}")

# ── Wykres słupkowy grupowany ───────────────────────────────────────────
arch_names = list(f1_before.keys())
x      = np.arange(len(arch_names))
width  = 0.38

fig, ax = plt.subplots(figsize=(10, 6))
bars_b = ax.bar(x - width/2, [f1_before[n] for n in arch_names],
                width, label='Domyślne (lr=1e-3, bs=32)', color='steelblue', alpha=0.8)
bars_a = ax.bar(x + width/2, [f1_after[n]  for n in arch_names],
                width, label='Optymalne (GridSearch DL)', color='coral', alpha=0.8)

for bar in bars_b:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, color='steelblue')
for bar in bars_a:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=10, color='coral')

for i, name in enumerate(arch_names):
    delta = f1_after[name] - f1_before[name]
    y_top = max(f1_before[name], f1_after[name]) + 0.04
    ax.annotate(f'Δ={delta:+.3f}', xy=(x[i], y_top), ha='center',
                fontsize=10, color='green' if delta >= 0 else 'red', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(arch_names, fontsize=12)
ax.set_ylabel('F1-macro (zbiór testowy)', fontsize=12)
ax.set_title('Wpływ tuningu hiperparametrów na modele DL\n(domyślne vs optymalne parametry)',
             fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, min(1.0, max(list(f1_after.values()) + list(f1_before.values())) + 0.15))
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../results/tuning_dl_before_after.png', bbox_inches='tight', dpi=150)
plt.show()

print("\nNajlepsze konfiguracje:")
for name in arch_names:
    cfg = dl_tune_results[name]['best_config']
    print(f"  {name:10s}: {cfg}")

---
## 9. Wpływ augmentacji danych na jakość klasyfikacji DL

Augmentacja jest szczególnie ważna dla sieci neuronowych – mają dużo parametrów
i są podatne na overfitting przy małych zbiorach danych.

Porównujemy trening CNN1D:
- **bez augmentacji** – oryginalne dane (wynik z sekcji 6)
- **z augmentacją** – zbiór 2× większy (5 technik losowo mieszane)

In [ ]:
import sys
sys.path.insert(0, '..')
from src.augmentation import augment_dataset

# ── Powiększ zbiór treningowy augmentacją ───────────────────────────────
print("Augmentacja danych treningowych dla DL...")
X_train_aug_dl, y_train_aug_dl = augment_dataset(
    X_train, y_train,
    techniques=['noise', 'shift', 'scale', 'wander', 'mask'],
    augment_factor=1,   # podwojenie
    fs=FS, seed=42, verbose=True,
)
print(f"Rozmiar: {len(X_train)} → {len(X_train_aug_dl)}")

# DataLoader z augmentowanymi danymi
best_bs = dl_tune_results['CNN1D']['best_config']['batch_size']
train_loader_aug = DataLoader(
    ECGDataset(X_train_aug_dl, y_train_aug_dl),
    batch_size=best_bs, shuffle=True, num_workers=0
)

# ── Trening CNN1D z augmentacją (te same optymalne hiperparametry) ────────
print(f"\nTrening CNN1D z augmentacją (lr={dl_tune_results['CNN1D']['best_config']['lr']:.0e}, "
      f"bs={best_bs}, {EPOCHS} epok)...")

cnn_aug        = CNN1D(num_channels=12, num_classes=len(SUPERCLASSES))
cnn_aug_trainer = Trainer(cnn_aug, device=DEVICE,
                           learning_rate=dl_tune_results['CNN1D']['best_config']['lr'])
hist_cnn_aug = cnn_aug_trainer.fit(train_loader_aug, val_loader, epochs=EPOCHS, verbose=True)

# ── Ewaluacja na zbiorze testowym ─────────────────────────────────────────
tl_test_aug = DataLoader(ECGDataset(X_test, y_test), batch_size=best_bs,
                          shuffle=False, num_workers=0)

y_true_aug, y_pred_aug = cnn_aug_trainer.predict(tl_test_aug)
f1_cnn_aug = f1_score(y_true_aug, y_pred_aug, average='macro', zero_division=0)
f1_cnn_no_aug = f1_after['CNN1D']   # wynik z sekcji 8 (optymalne, bez augmentacji)

print(f"\nCNN1D bez augmentacji: F1={f1_cnn_no_aug:.4f}")
print(f"CNN1D z augmentacją:   F1={f1_cnn_aug:.4f}  Δ={f1_cnn_aug - f1_cnn_no_aug:+.4f}")

# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA: Krzywe uczenia + porównanie F1
# ═══════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Krzywe uczenia: bez augmentacji vs z augmentacją
axes[0].plot(tuned_histories['CNN1D']['val_acc'], 'steelblue', linewidth=2,
             label='Bez augmentacji')
axes[0].plot(hist_cnn_aug['val_acc'],             'coral',     linewidth=2,
             label='Z augmentacją (×2)')
axes[0].set_xlabel('Epoka')
axes[0].set_ylabel('Val Accuracy')
axes[0].set_title('CNN1D: krzywe uczenia\nbez vs z augmentacją', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Słupki porównawcze
bars = axes[1].bar(['Bez\naugmentacji', 'Z augmentacją\n(×2)'],
                   [f1_cnn_no_aug, f1_cnn_aug],
                   color=['steelblue', 'coral'], alpha=0.85, width=0.4)
for bar, val in zip(bars, [f1_cnn_no_aug, f1_cnn_aug]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

delta = f1_cnn_aug - f1_cnn_no_aug
axes[1].set_ylim(0, max(f1_cnn_no_aug, f1_cnn_aug) + 0.12)
axes[1].set_ylabel('F1-macro (zbiór testowy)', fontsize=12)
axes[1].set_title(f'CNN1D: wpływ augmentacji\nΔ = {delta:+.4f}', fontweight='bold')
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Augmentacja danych – CNN1D', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/augmentation_cnn1d_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

---
## 10. Interpretowalność modeli – Mapy istotności (Saliency Maps)

Mapy istotności odpowiadają na pytanie:
**"Które fragmenty sygnału EKG najbardziej wpłynęły na decyzję modelu?"**

Zaimplementowane metody (z `src/saliency.py`):

| Metoda | Formuła | Opis |
|--------|---------|------|
| **Vanilla Saliency** | `∣∂score/∂x∣` | Gradient wyjścia względem wejścia |
| **Gradient × Input** | `∣∂score/∂x · x∣` | Gradient × amplituda – uwzględnia wartość sygnału |
| **Attention Weights** | wagi softmax z BiLSTM | Bezpośrednia wizualizacja na czym skupia się LSTM |

Wizualizacje:
1. Saliency na pojedynczym sygnale EKG – nakładka na sygnał
2. Uśrednione mapy istotności per klasa diagnostyczna
3. Heatmapa ważności odprowadzeń (12 leads × 5 klas)
4. Wagi uwagi (attention) z BiLSTM

In [ ]:
import sys
sys.path.insert(0, '..')
from src.saliency import (
    vanilla_saliency, gradient_x_input,
    get_attention_weights, compute_class_saliency, lead_importance,
)
from src.data_loader import SUPERCLASSES

LEAD_NAMES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Używamy najlepszego modelu CNN1D (po tuningu, bez augmentacji)
# oraz modelu ResNet1D i BiLSTM z sekcji 7
model_cnn    = tuned_models['CNN1D']
model_resnet = tuned_models['ResNet1D']
model_lstm   = tuned_models['BiLSTM']

print("Modele gotowe do analizy interpretowalności.")
print(f"Zbiór testowy: {X_test.shape[0]} próbek, {len(SUPERCLASSES)} klas")
print(f"Klasy: {list(SUPERCLASSES)}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 1: Saliency na pojedynczym sygnale EKG
# ═══════════════════════════════════════════════════════════════════════════
# Pokazujemy sygnał EKG (odprowadzenie II) z nałożoną mapą istotności.
# Kolory tła odpowiadają "ważności" każdej próbki:
#   ciemniejsze = bardziej wpłynęło na decyzję modelu
#
# Wizualizujemy dwie metody obok siebie (Vanilla vs Gradient×Input)
# dla jednej próbki każdej klasy.

fig, axes = plt.subplots(len(SUPERCLASSES), 2, figsize=(16, 3 * len(SUPERCLASSES)))
t = np.arange(X_test.shape[1]) / FS   # oś czasu [s]
lead_vis = 1                            # odprowadzenie II – najczytelniejsze klinicznie

for row, cls_name in enumerate(SUPERCLASSES):
    # Znajdź pierwszą próbkę tej klasy w zbiorze testowym
    cls_idx = np.where(y_test == le.transform([cls_name])[0])[0]
    if len(cls_idx) == 0:
        continue
    sample_idx  = cls_idx[0]
    x_sample    = X_test[sample_idx]          # (n_samples, 12)
    true_label  = le.inverse_transform([y_test[sample_idx]])[0]
    target_cls  = y_test[sample_idx]

    for col, (method_name, sal_fn) in enumerate([
        ('Vanilla Saliency (∣∇∣)',        vanilla_saliency),
        ('Gradient × Input (∣∇·x∣)',      gradient_x_input),
    ]):
        ax = axes[row, col]
        sal_map = sal_fn(model_cnn, x_sample, int(target_cls), device=DEVICE)
        # sal_map: (n_leads, n_samples) → weź odprowadzenie lead_vis
        sal_lead = sal_map[lead_vis]  # (n_samples,)

        # Normalizacja mapy do [0, 1] dla kolorowania
        sal_norm = (sal_lead - sal_lead.min()) / (sal_lead.max() - sal_lead.min() + 1e-10)

        # Sygnał EKG (odprowadzenie II)
        ecg_lead = x_sample[:, lead_vis]

        # Kolorowe tło – fill_between segmentami
        for i in range(len(t) - 1):
            ax.axvspan(t[i], t[i+1], alpha=float(sal_norm[i]) * 0.6,
                       color='red', linewidth=0)

        # Sygnał na wierzchu
        ax.plot(t, ecg_lead, color='black', linewidth=0.8, zorder=5)

        if col == 0:
            ax.set_ylabel(f'Klasa: {true_label}', fontsize=10, fontweight='bold')
        ax.set_title(f'{method_name}', fontsize=9)
        ax.set_xlim(t[0], t[-1])
        ax.grid(True, alpha=0.2)

        if row == len(SUPERCLASSES) - 1:
            ax.set_xlabel('Czas [s]', fontsize=9)

plt.suptitle('Mapy istotności – CNN1D (odprowadzenie II)\n'
             'Czerwone tło: obszary kluczowe dla decyzji modelu',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/saliency_single_sample.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 2: Uśrednione mapy istotności per klasa (heatmapa czas × odprow.)
# ═══════════════════════════════════════════════════════════════════════════
# Dla każdej klasy diagnostycznej obliczamy ŚREDNIĄ mapę istotności
# ze wszystkich próbek tej klasy ze zbioru testowego.
# Heatmapa (12 odprowadzeń × n_próbek_czasu) pokazuje:
#   - Które regiony czasowe są typowo ważne dla danej klasy
#   - Które z 12 odprowadzeń wnoszą więcej informacji

print("Obliczanie uśrednionych map istotności (Gradient×Input) per klasa...")
avg_saliency_maps = compute_class_saliency(
    model_cnn, X_test, y_test,
    method='gradient_x_input',
    device=DEVICE,
    n_classes=len(SUPERCLASSES),
)
print("✓ Gotowe.")

# ── Heatmapa: czas × odprowadzenia dla każdej klasy ─────────────────────
fig, axes = plt.subplots(1, len(SUPERCLASSES), figsize=(18, 5))

for ax, (cls_idx, cls_name) in enumerate(zip(range(len(SUPERCLASSES)), SUPERCLASSES)):
    sal = avg_saliency_maps.get(cls_idx)
    if sal is None:
        axes[ax].set_visible(False)
        continue

    # Wygładź mapę czasową (ruchoma średnia 10 próbek) dla czytelności
    from scipy.ndimage import uniform_filter1d
    sal_smooth = uniform_filter1d(sal, size=10, axis=1)

    # Normalizacja do [0,1]
    sal_norm = (sal_smooth - sal_smooth.min()) / (sal_smooth.max() - sal_smooth.min() + 1e-10)

    # Redukcja czasowa do 200 punktów dla szybkości rysowania
    step = max(1, sal_norm.shape[1] // 200)
    sal_reduced = sal_norm[:, ::step]

    im = axes[ax].imshow(
        sal_reduced, cmap='hot', aspect='auto',
        origin='upper', vmin=0, vmax=1,
    )
    axes[ax].set_yticks(range(12))
    axes[ax].set_yticklabels(LEAD_NAMES, fontsize=8)
    axes[ax].set_xlabel('Czas →', fontsize=9)
    axes[ax].set_xticks([])
    axes[ax].set_title(f'Klasa: {cls_name}', fontweight='bold', fontsize=10)

    plt.colorbar(im, ax=axes[ax], shrink=0.8, label='Istotność')

plt.suptitle('Uśrednione mapy istotności (Gradient×Input) – CNN1D\n'
             'Jaśniejsze = ważniejsze dla decyzji | Wiersze = 12 odprowadzeń EKG',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../results/saliency_class_heatmaps.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 3: Heatmapa ważności odprowadzeń (12 leads × 5 klas)
# ═══════════════════════════════════════════════════════════════════════════
# Redukujemy mapę (12, n_samples) → (12,) przez sumę po czasie.
# Wynikowa heatmapa odpowiada na pytanie:
# "Które odprowadzenia są najważniejsze dla każdej klasy diagnostycznej?"
#
# Klinicznie sensowne wzorce:
#   MI (zawał) → V1-V4 ważniejsze (przedni ściana)
#   CD (blok)  → I, aVR mogą być ważniejsze (oś elektryczna)
#   HYP (przerost) → V5-V6 (ściana boczna)

importance_matrix = np.zeros((12, len(SUPERCLASSES)))   # (n_leads, n_classes)

for cls_idx, cls_name in enumerate(SUPERCLASSES):
    sal = avg_saliency_maps.get(cls_idx)
    if sal is not None:
        importance_matrix[:, cls_idx] = lead_importance(sal)

fig, ax = plt.subplots(figsize=(10, 7))
im = ax.imshow(importance_matrix, cmap='YlOrRd', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label='Ważność odprowadzenia (znorm. do [0,1])')

ax.set_xticks(range(len(SUPERCLASSES)))
ax.set_xticklabels(list(SUPERCLASSES), fontsize=12)
ax.set_yticks(range(12))
ax.set_yticklabels(LEAD_NAMES, fontsize=11)
ax.set_xlabel('Klasa diagnostyczna', fontsize=12)
ax.set_ylabel('Odprowadzenie EKG', fontsize=12)

# Adnotacje wartości
for i in range(12):
    for j in range(len(SUPERCLASSES)):
        val = importance_matrix[i, j]
        ax.text(j, i, f'{val:.2f}',
                ha='center', va='center', fontsize=9,
                color='white' if val > 0.65 else 'black', fontweight='bold')

ax.set_title('Ważność odprowadzeń per klasa diagnostyczna\n'
             '(CNN1D, Gradient×Input, uśrednione)', fontweight='bold', fontsize=13)

plt.tight_layout()
plt.savefig('../results/saliency_lead_importance_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()

# Wypisz najważniejsze odprowadzenie per klasa
print("\nNajważniejsze odprowadzenie per klasa diagnostyczna:")
for cls_idx, cls_name in enumerate(SUPERCLASSES):
    top_lead_idx = np.argmax(importance_matrix[:, cls_idx])
    print(f"  {cls_name}: {LEAD_NAMES[top_lead_idx]} (ważność={importance_matrix[top_lead_idx, cls_idx]:.3f})")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 4: Wagi uwagi (Attention Weights) z BiLSTM
# ═══════════════════════════════════════════════════════════════════════════
# Mechanizm uwagi BiLSTM bezpośrednio przypisuje "ważność" każdej chwili czasu.
# Suma wag = 1 (rozkład prawdopodobieństwa).
# Piki uwagi wskazują na fragmeny sygnału kluczowe dla klasyfikacji
# (typowo obszary QRS, odcinek ST lub fala T).

fig, axes = plt.subplots(len(SUPERCLASSES), 1, figsize=(14, 3.5 * len(SUPERCLASSES)), sharex=True)
t = np.arange(X_test.shape[1]) / FS

for row, cls_name in enumerate(SUPERCLASSES):
    cls_idx_val = np.where(y_test == le.transform([cls_name])[0])[0]
    if len(cls_idx_val) == 0:
        continue
    x_sample = X_test[cls_idx_val[0]]

    # Oblicz wagi uwagi
    attn_weights = get_attention_weights(model_lstm, x_sample, device=DEVICE)

    ax = axes[row]
    ax2 = ax.twinx()   # drugi wykres na tej samej osi X

    # Sygnał EKG (odprowadzenie II) – na tle
    ax.plot(t, x_sample[:, 1], color='steelblue', linewidth=0.8, alpha=0.7, label='EKG (odprow. II)')
    ax.set_ylabel('Amplituda EKG', fontsize=9, color='steelblue')
    ax.tick_params(axis='y', labelcolor='steelblue')

    # Wagi uwagi (wygładzone 5-próbkową średnią ruchomą)
    from scipy.ndimage import uniform_filter1d
    attn_smooth = uniform_filter1d(attn_weights, size=5)
    ax2.fill_between(t, 0, attn_smooth, color='red', alpha=0.35, label='Uwaga (attention)')
    ax2.set_ylabel('Waga uwagi', fontsize=9, color='red')
    ax2.tick_params(axis='y', labelcolor='red')

    ax.set_title(f'BiLSTM Attention – klasa: {cls_name}', fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.2)

    # Oznacz szczyt uwagi
    peak_t = t[np.argmax(attn_smooth)]
    ax2.axvline(peak_t, color='darkred', linestyle='--', linewidth=1, alpha=0.8,
                label=f'Szczyt: t={peak_t:.2f}s')
    ax2.legend(loc='upper right', fontsize=8)

axes[-1].set_xlabel('Czas [s]', fontsize=11)
plt.suptitle('BiLSTM – Attention Weights na sygnale EKG\n'
             'Czerwony obszar: na czym skupia się sieć podejmując decyzję',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/saliency_bilstm_attention.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# WIZUALIZACJA 5: Porównanie Vanilla vs Gradient×Input vs Attention (CNN1D)
# ═══════════════════════════════════════════════════════════════════════════
# Dla jednej próbki (klasa MI – zawał, klinicznie interesująca)
# pokazujemy wszystkie 3 metody interpretowalności obok siebie.

# Wybierz próbkę klasy MI (lub pierwszą dostępną klasę)
target_cls_name = 'MI' if 'MI' in SUPERCLASSES else list(SUPERCLASSES)[0]
cls_idx_mi  = le.transform([target_cls_name])[0]
mi_indices  = np.where(y_test == cls_idx_mi)[0]
x_mi        = X_test[mi_indices[0]]    # (n_samples, 12)

t = np.arange(x_mi.shape[0]) / FS
lead_vis = 1  # odprowadzenie II

# Oblicz wszystkie 3 mapy
sal_vanilla = vanilla_saliency(model_cnn,    x_mi, cls_idx_mi, device=DEVICE)
sal_gxi     = gradient_x_input(model_cnn,   x_mi, cls_idx_mi, device=DEVICE)
sal_gxi_rn  = gradient_x_input(model_resnet, x_mi, cls_idx_mi, device=DEVICE)
attn_bilstm = get_attention_weights(model_lstm, x_mi, device=DEVICE)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

def plot_saliency_overlay(ax, ecg, sal_1d, title, color='red'):
    """Rysuje sygnał EKG z nałożoną mapą istotności jako kolorowe tło."""
    sal_n = (sal_1d - sal_1d.min()) / (sal_1d.max() - sal_1d.min() + 1e-10)
    for i in range(len(t) - 1):
        ax.axvspan(t[i], t[i+1], alpha=float(sal_n[i]) * 0.65, color=color, linewidth=0)
    ax.plot(t, ecg[:, lead_vis], color='black', linewidth=0.9, zorder=5)
    ax.set_ylabel('Amplituda', fontsize=9)
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.grid(True, alpha=0.2)

# 1. Oryginał
axes[0].plot(t, x_mi[:, lead_vis], color='black', linewidth=0.9)
axes[0].set_title(f'Sygnał EKG – klasa {target_cls_name} (odprowadzenie II)',
                  fontweight='bold', fontsize=10)
axes[0].grid(True, alpha=0.2)

# 2. Vanilla Saliency (CNN1D)
plot_saliency_overlay(axes[1], x_mi, sal_vanilla[lead_vis],
                      'Vanilla Saliency – CNN1D (∣∇∣)', color='red')

# 3. Gradient × Input (ResNet1D – porównanie architectury)
plot_saliency_overlay(axes[2], x_mi, sal_gxi_rn[lead_vis],
                      'Gradient × Input – ResNet1D (∣∇·x∣)', color='darkorange')

# 4. BiLSTM Attention
axes[3].plot(t, x_mi[:, lead_vis], color='steelblue', linewidth=0.8, alpha=0.7)
from scipy.ndimage import uniform_filter1d
attn_s = uniform_filter1d(attn_bilstm, size=5)
ax3b = axes[3].twinx()
ax3b.fill_between(t, 0, attn_s, color='green', alpha=0.4)
ax3b.set_ylabel('Waga uwagi', color='green', fontsize=9)
ax3b.tick_params(axis='y', labelcolor='green')
axes[3].set_title('BiLSTM Attention Weights', fontweight='bold', fontsize=10)
axes[3].grid(True, alpha=0.2)

axes[-1].set_xlabel('Czas [s]', fontsize=11)

plt.suptitle(f'Porównanie metod interpretowalności – klasa {target_cls_name}\n'
             'Kolorowe tło = obszary ważne dla decyzji modelu',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/saliency_methods_comparison.png', bbox_inches='tight', dpi=150)
plt.show()